# Orbit Wars — Model Mimarisi Teorik Açıklama

Bu notebook, Orbit Wars projesindeki **Set Transformer** tabanlı policy/value ağının
mimarisini teorik olarak açıklar. Hem PPO policy ağı (`OrbitWarsPolicy`, `PlanetOnlyOrbitWarsPolicy`)
hem de DQN ağı (`OrbitWarsQNet`) aynı encoder'ı paylaşır.

## 1. Genel Mimari

```
┌─────────────────────────────────────────────────────────────────┐
│                       OBSERVATION                               │
│  planet_feats (48×33)  fleet_feats (64×11)  global_feats (14)  │
│  edge_bias (113×113×6)  masks                                  │
└──────────────────────────┬──────────────────────────────────────┘
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                   SET TRANSFORMER ENCODER                       │
│                                                                 │
│  Token İnşası:  LinearProj + TypeEmbed → (B, T, d_model)       │
│  Edge Bias:     Linear(edge_dim → n_heads) → (B, H, T, T)      │
│                                                                 │
│  ┌───────────────────────────────────────────────────────────┐ │
│  │  TransformerBlock × N                                      │ │
│  │  ┌─────────────────┐   ┌─────────────────────────────────┐ │ │
│  │  │ BiasedMHA       │   │ FFN: Linear→GELU→Linear         │ │ │
│  │  │ QKᵀ/√d + bias   │   │ (4× expansion)                  │ │ │
│  │  └─────────────────┘   └─────────────────────────────────┘ │ │
│  │  Pre-norm + residual                                      │ │
│  └───────────────────────────────────────────────────────────┘ │
│                         × N (4 veya 6)                          │
│                         → LayerNorm                             │
└──────────────────────────┬──────────────────────────────────────┘
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                         HEADS                                   │
│                                                                 │
│  planet_tok[:, :P]  ──→ src_head    → (B, P+1)  source logits  │
│  planet_tok[:, :P]  ──→ tgt_query   → (B, P, P) target logits  │
│                    └──→ tgt_key                                 │
│  planet_tok[:, :P]  ──→ ship_head   → (B, P, 5) ship bin logits│
│  global_tok[:, -1]  ──→ value_head  → (B,)     state value     │
│  planet_tok[:, :P]  ──→ aux_owner   → (B, P, 3) aux. head      │
└─────────────────────────────────────────────────────────────────┘
```

## 2. Token Düzeni

Set Transformer, değişken sayıda gezegen ve filoyu **sabit boyutlu** bir dizi olarak temsil eder.
Gerçek sayıdan az olan slot'lar padding ile doldurulur, maskelenir.

### 2.1. Token Yapısı

| Token Tipi | Adet | Feature Dim | Projeksiyon | Type ID |
|---|---:|---:|---|---|
| Gezegen | `P_max = 48` | 33 | `planet_proj: Linear(33, d)` | 0 |
| Filo | `F_max = 64` | 11 | `fleet_proj: Linear(11, d)` | 1 |
| Global (CLS) | 1 | 14 | `global_proj: Linear(14, d)` | 2 |
| **Toplam** | **T = 113** | — | — | — |

```
Token dizilimi:
[P₀, P₁, ..., P₄₇, F₀, F₁, ..., F₆₃, GLOBAL]
 ├─── 48 gezegen ───┤├──── 64 filo ────┤├─ 1 ─┤
```

### 2.2. Token İnşası (Kod)

```python
planet_tok = self.planet_proj(planet_feats)   # (B, 48, 33) → (B, 48, d)
fleet_tok  = self.fleet_proj(fleet_feats)     # (B, 64, 11) → (B, 64, d)
global_tok = self.global_proj(global_feats)   # (B, 14)    → (B, 1, d)

x = cat([planet_tok, fleet_tok, global_tok], dim=1)  # (B, 113, d)
x = x + self.type_embed(type_ids)                    # tip embedding ekle
```

**Neden ayrı projeksiyonlar?** Gezegen, filo ve global token'lar farklı dağılımlardan
gelir (farklı ölçekler, farklı anlamlar). Aynı `Linear` katmanını paylaşmak yerine
her tip kendi projeksiyonunu öğrenir. Bu, heterojen set'ler için kritik bir tasarım
kararıdır.

## 3. Edge Bias — Geometrik Önbilgi

Set Transformer'ı standart Transformer'dan ayıran en önemli mekanizma **edge bias**'tır.
Her token çifti `(i, j)` için hesaplanan geometrik özellikler, attention skorlarına
eklenir.

### 3.1. Edge Bias Kanalları

| Kanal | Formül | Normalizasyon | Anlam |
|---:|---|---|---|
| 0 | `‖posᵢ − posⱼ‖` | `/100` | Öklid mesafesi |
| 1 | `xⱼ − xᵢ` | `/100` | x yönü |
| 2 | `yⱼ − yᵢ` | `/100` | y yönü |
| 3 | `sin(φⱼ − φᵢ)` | — | Güneş etrafı açı farkı (sin) |
| 4 | `cos(φⱼ − φᵢ)` | — | Güneş etrafı açı farkı (cos) |
| 5 | `min_dist(seg, sun) < R+0.5` | — | Direkt yol güneşi kesiyor mu? |

### 3.2. Neden `sin/cos(Δφ)`?

Aynı yörüngedeki iki gezegen için `Δφ` **zamanla değişmez** (ikisi de aynı `ω` ile döner).
Bu sayede model, rotasyonel simetriden bağımsız, stabil bir yörüngesel ilişki sinyali alır.

```
Zaman: t=0          Zaman: t=10         Zaman: t=20
    Pᵢ                 Pᵢ                  Pᵢ
     ●──────● Pⱼ        ●──────● Pⱼ         ●──────● Pⱼ
   Δφ = 45°           Δφ = 45°            Δφ = 45°
   (değişmez!)         (değişmez!)          (değişmez!)
```

### 3.3. Edge Bias → Attention Bias

```python
# edge_bias: (B, T, T, 6)
attn_bias = self.edge_proj(edge_bias)      # (B, T, T, n_heads)
attn_bias = attn_bias.permute(0, 3, 1, 2)  # (B, n_heads, T, T)
```

Her attention head'i, 6 kanallı edge bilgisinden kendi `n_heads` kanallı bias'ını
öğrenir. Bu sayede farklı head'ler farklı geometrik ilişkilere odaklanabilir
(ör: Head-0 mesafeye, Head-1 yörünge ilişkisine, Head-2 güneş engeline...).

## 4. Biased Multi-Head Attention

Standart MHA'dan tek farkı, attention skorlarına **additive bias** eklenmesidir.

### 4.1. Matematiksel Formül

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \mathbf{B}\right) \cdot V$$

Burada $\mathbf{B} \in \mathbb{R}^{H \times T \times T}$ her head için ayrı bir bias matrisidir.

### 4.2. Standart MHA vs Biased MHA

| Özellik | Standart MHA | Biased MHA |
|---|---|---|
| Attention skoru | `QKᵀ / √d` | `QKᵀ / √d + B` |
| Pozisyonel encoding | Learned / sinusoidal | **Geometrik edge bias** |
| Uzaysal ilişki | Sıfırdan öğrenilir | **Doğrudan enjekte edilir** |
| Permütasyon invaryansı | Var (padding mask ile) | Var |

### 4.3. Kod Akışı

```python
Q, K, V = split(W_qkv(x))              # (B, H, T, d_h)
scores = Q @ Kᵀ / √d_h                 # (B, H, T, T)
scores = scores + attn_bias            # ← GEOMETRİK ÖNBİLGİ
scores = masked_fill(scores, pad, -∞)  # padding token'ları maskele
attn = softmax(scores)
out = attn @ V
```

## 5. Transformer Block (Pre-Norm)

```
┌────────────────────────────────────────┐
│         TRANSFORMER BLOCK              │
│                                        │
│  x ──→ LayerNorm ──→ BiasedMHA ──→ + ─│──→ LayerNorm ──→ FFN ──→ + ──→
│       (pre-norm)    (edge bias)    ↑   │   (pre-norm)            ↑
│       └────────────────────────────┘   │   └─────────────────────┘
│            (residual)                  │        (residual)
└────────────────────────────────────────┘

FFN: Linear(d → 4d) → GELU → Linear(4d → d)
```

| Bileşen | Detay |
|---|---|
| Normalizasyon | **Pre-norm** (attention/FFN öncesi LayerNorm) |
| Aktivasyon | **GELU** (Gaussian Error Linear Unit) |
| FFN genişleme | **4×** (`ff_mult=4`) |
| Dropout | `0.0` (şu an kapalı) |
| Residual | Her iki alt katmanda |

## 6. Aksiyon Faktorizasyonu

Model, bileşik aksiyonu üç bağımsız seçime ayırır (Branching / faktörize policy):

$$p(a \mid s) = p_\text{src}(\text{slot} \mid s) \cdot p_\text{tgt}(\text{slot} \mid s, \text{src}) \cdot p_\text{ship}(\text{bin} \mid s, \text{src})$$

### 6.1. Source Head

| Özellik | Değer |
|---|---|
| Çıktı | `(B, P+1)` — her gezegen için 1 skor + NO-OP slotu |
| Mimari | `Linear(d, 1)` → squeeze → `cat([masked_scores, noop_param])` |
| Maske | `source_mask`: sadece kendi gezegenlerim ve ≥1 gemisi olanlar |
| NO-OP | Öğrenilen skaler parametre (`nn.Parameter`) |

### 6.2. Target Head (Pointer-style)

| Özellik | Değer |
|---|---|
| Çıktı | `(B, P, P)` — `q(src) · k(tgt)ᵀ / √d` |
| Mimari | Query: `Linear(d, d)`, Key: `Linear(d, d)` → scaled dot-product |
| Maske | Self-target (`i=j`) + padding gezegenler |

```
q = tgt_query(planet_tok)     # (B, P, d)
k = tgt_key(planet_tok)       # (B, P, d)
tgt_logits = q @ kᵀ / √d      # (B, P, P)
tgt_logits[i, i] = -∞         # kendine fırlatma yasak
```

### 6.3. Ship Head

| Özellik | Değer |
|---|---|
| Çıktı | `(B, P, 5)` — her gezegen için 5 bin skoru |
| Mimari | `Linear(d, 5)` |
| Bin'ler | `[10%, 25%, 50%, 75%, 100%]` (mevcut gemilerin yüzdesi) |

### 6.4. Aksiyon Seçimi (Greedy)

```
src*  = argmax src_logits[masked]     → hangi gezegenimden?
tgt*  = argmax tgt_logits[src*]       → hangi gezegene?
ship* = argmax ship_logits[src*]      → gemilerin yüzde kaçı?

Gerçek gemi sayısı = int(available_ships * ship_bins[ship*])
Fırlatma açısı     = navigation.compute_launch(src, tgt, ships)
```

## 7. Value Head

State value fonksiyonu $V(s)$:

| Varyant | Girdi | Mimari |
|---|---|---|
| **Base** (`OrbitWarsPolicy`) | Sadece `global_tok` | `Linear(d, d) → GELU → Linear(d, 1)` |
| **PlanetOnly** (yeni) | `cat([global_tok, mean(planet_tok)])` | `Linear(2d, d) → GELU → Linear(d, 1)` |

PlanetOnly'de `mean(planet_tok)` eklenmesinin nedeni: global token oyunun
özetini taşır, ama gezegenlerin durumunu doğrudan value tahminine katmak
daha zengin bir sinyal sağlar.

## 8. Yardımcı (Auxiliary) Head

PlanetOnly modelinde eklenen yardımcı tahmin başlığı:

| Özellik | Değer |
|---|---|
| Çıktı | `(B, P, 3)` — her gezegen için 3 sınıf logiti |
| Sınıflar | `[mine, enemy, neutral]` — bir sonraki turda beklenen sahiplik |
| Mimari | `Linear(d, 3)` |
| Amaç | Temsiliyet öğrenmeyi (representation learning) hızlandırmak |
| Loss | Cross-entropy (PPO loss'una ek olarak, henüz bağlanmadı) |

## 9. Model Varyantları

### 9.1. Karşılaştırma Tablosu

| Özellik | `OrbitWarsPolicy` | `PlanetOnlyOrbitWarsPolicy` | `OrbitWarsQNet` |
|---|---|---|---|
| **Token sayısı (T)** | 113 | 49 | 113 |
| **Filo token'ları** | ✅ 64 slot | ❌ | ✅ 64 slot |
| **Type embed sınıf** | 3 | 2 | 3 |
| **d_model (default)** | 128 | **256** | 128 |
| **n_layers (default)** | 4 | **6** | 4 |
| **Value head girdi** | global | global + mean(planet) | global (dueling V) |
| **Auxiliary head** | ❌ | ✅ `Linear(d,3)` | ❌ |
| **Dueling** | ❌ | ❌ | ✅ (opsiyonel) |
| **Çıktı** | Policy + Value | Policy + Value + Aux | Q-değerleri |
| **Kullanım** | PPO (tam) | PPO (hafif) | DQN |

### 9.2. PlanetOnly Tasarım Kararı

Filo token'larının kaldırılmasının nedeni:
- Filo bilgisi zaten **gezegen özelliklerinde** ETA bucket'ları olarak kodlanmıştır
  (hangi gezegene kaç gemi geliyor, ne zaman)
- Global özelliklerde toplam hava üstünlüğü (`air_advantage`) vardır
- Daha az token = daha hızlı encoding, daha az overfitting riski
- **Tradeoff**: Tekil filo bilgisi (tam konum, açı) kaybolur

## 10. OrbitWarsQNet — Branching Dueling DQN

DQN varyantı aynı Set Transformer encoder'ı kullanır, ancak head'ler **Q-değeri** üretir.

### 10.1. Branching DQN Formülasyonu

$$Q_\text{total}(s, a) = Q_\text{src}(s, \text{src}) + \mathbf{1}[\text{src} \neq \text{NOOP}] \cdot \big(Q_\text{tgt}(s, \text{src}, \text{tgt}) + Q_\text{ship}(s, \text{src}, \text{ship})\big)$$

### 10.2. Dueling Ayrıştırması

$$Q_b(s, a) = V(s) + A_b(s, a) - \frac{1}{|\mathcal{A}_b|}\sum_{a'} A_b(s, a')$$

- $V(s)$: Tek bir shared state value (global token'dan)
- $A_b(s, a)$: Branch başına avantaj fonksiyonu
- Mean çıkarma: Identifiability için (avantajın ortalaması sıfır)

### 10.3. Aksiyon Seçimi

```
src*  = argmax Q_src(s, a)       # bağımsız argmax
tgt*  = argmax Q_tgt(s, src*, a) # src* koşullu
ship* = argmax Q_ship(s, src*, a) # src* koşullu
```

Faktörize yapı sayesinde $\max_a Q(s,a)$ **exact** hesaplanabilir
(her branch bağımsız maksimize edilir, toplamalı yapı garantiler).

## 11. Parametre Sayıları

| Model | d_model | n_layers | Parametre |
|---:|---:|---:|---:|
| `OrbitWarsPolicy` | 128 | 4 | ~1.3M |
| `PlanetOnlyOrbitWarsPolicy` (eski) | 128 | 4 | ~1.3M |
| `PlanetOnlyOrbitWarsPolicy` (**yeni**) | 256 | 6 | **~5.0M** |
| `OrbitWarsQNet` | 128 | 4 | ~1.3M |
| `OrbitWarsQNet` (dueling) | 128 | 4 | ~1.3M |

Yeni PlanetOnly modeli ~3.8× daha fazla parametreye sahip, ancak token sayısı
az (49 vs 113) olduğu için inference süresi hala yönetilebilir.

## 12. Observation → Model Veri Akışı

```
env.step() → raw observation (dict)
     │
     ▼
build_observation(env_obs, player, omega, max_turns)
     │
     ├─→ parse_planets(env_obs)       → Planet objects
     ├─→ parse_fleets(env_obs)        → Fleet objects
     ├─→ predict_all_fleet_targets()  → her filo için (tgt_id, eta)
     ├─→ incoming_by_planet           → ETA bucket dağılımı
     │
     ├─→ _planet_features()  × P  → (P_max, 33)
     ├─→ _fleet_features()   × F  → (F_max, 11)
     ├─→ _global_features()       → (14,)
     ├─→ edge_bias                → (T, T, 6)
     ├─→ source_mask              → (P_max,) bool
     │
     ▼
Observation dataclass → pack_observations() → batch dict
     │
     ▼
policy.forward(**batch) → PolicyOutput
     │
     ├─→ src_logits   (B, P+1)
     ├─→ tgt_logits   (B, P, P)
     ├─→ ship_logits  (B, P, 5)
     ├─→ value        (B,)
     └─→ aux_owner    (B, P, 3)   [opsiyonel]
```

## 13. Tasarım Prensipleri Özeti

| Prensip | Uygulama |
|---|---|
| **Permütasyon invaryansı** | Set Transformer — gezegen/filo sıralaması önemsiz |
| **Rotasyonel simetri** | Kutupsal koordinatlar (r, θ) + sin/cos(Δφ) edge bias |
| **Geometrik önbilgi** | Edge bias → attention'a additive bias |
| **Değişken sayıda nesne** | Padding + maskeleme (P_max=48, F_max=64) |
| **Heterojen token'lar** | Tip başına ayrı projeksiyon + type embedding |
| **Faktörize aksiyon** | src → tgt → ship (3 bağımsız seçim) |
| **Ekonomik sinyal** | capture_value, defense_deficit, air_advantage |
| **Temsiliyet öğrenme** | Auxiliary owner prediction head |